# In Class Activity April 14th, 2026

### Importing libraries, preparing data, initial EDA

In [1]:
# importing libraries (feel free to add more if you want to explore other things)
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, RepeatedStratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import optuna

In [2]:
# importing data
adult = pd.read_csv('adult.csv')
adult.head(20)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K
5,34,Private,198693,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K
6,29,?,227026,HS-grad,9,Never-married,?,Unmarried,Black,Male,0,0,40,United-States,<=50K
7,63,Self-emp-not-inc,104626,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,>50K
8,24,Private,369667,Some-college,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,<=50K
9,55,Private,104996,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,<=50K


In [12]:
# initial EDA with sweetviz
report = sv.analyze(adult)
report.show_html('sweet_report.html')

# you are welcome to replace this cell with your own EDA, but make sure to include
# some visualizations and insights about the data

Done! Use 'show' commands to display/save.   |██████████| [100%]   00:00 -> (00:00 left)

Report sweet_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


### In the markdown cell below describe what you learned from your EDA and how it will inform your modeling decisions





What we found during the EDA is unlike many data sets that cotain things like N/A when a cell is missing data, this dataset has "?" which will confuse many model as they will than sense that features are str instread of numerical. it also means that the outputs of the visualizations have a caterogy for ?, which doesnt help us with determining distributions. native contry is anothe feature that should be closely inspected as most of the values fall under an other value, idicating lots of missing values or many distinct values. 

### Data Preprocessing (minimal) and Baseline Model

In [3]:
# data cleaning and preprocessing

# changing ? to NaN
adult = adult.replace('?', np.nan)

#education and education num are redundant, so we can drop one of them
adult = adult.drop('education', axis=1)

# target variable is income with 2 levels, so we can encode it as 0 and 1
adult['income'] = adult['income'].apply(lambda x: 1 if x == '>50K' else 0)

# change dtype categorical variables to category
categorical_cols = adult.select_dtypes(include='object').columns
adult[categorical_cols] = adult[categorical_cols].astype('category')


adult.head(20)

<positron-console-cell-3>:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.


,age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0
1,38,Private,89814,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,0
2,28,Local-gov,336951,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,1
3,44,Private,160323,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,1
4,18,NaN,103497,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,0
5,34,Private,198693,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,0
6,29,NaN,227026,9,Never-married,NaN,Unmarried,Black,Male,0,0,40,United-States,0
7,63,Self-emp-not-inc,104626,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,1
8,24,Private,369667,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,0
9,55,Private,104996,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,0


In [4]:
# defining features and target variable
X = adult.drop('income', axis=1)
y = adult['income']

# splitting data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, 
                                                    random_state=42, stratify=y)

In [5]:
# building xgboost default model and evaluating with stratified k-fold cross validation
xgb_cv = XGBClassifier(enable_categorical=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# each fold done gets an equal value of 0 and 1 
cv_scores = cross_val_score(xgb_cv, X, y, cv=skf, scoring='f1')
print(f'Cross-validated F1 scores: {cv_scores}')
print(f'Mean F1 score: {cv_scores.mean()}') 


Cross-validated F1 scores: [0.70680507 0.70892566 0.70898981 0.72424942 0.71086556]
Mean F1 score: 0.7119671046056588


### Use the markdown cell below to describe your baseline model performance and how you will try to improve it

Looking at the baseline model, we do not explore any of the hyperparamters and these can be tuned to help improve performance, another way we can improve it is by using different boosting models such is LightGLM or CatBoost and see if those have an impact on performance. 

### Model feature exploration

In the code cell below, explore different features of XGBoost and how they work (e.g. scale_pos_weight, max_depth, learning_rate).
Use stratified k-fold cross or repeated stratified k-fold cross validation with your model building. 
You should explore at least 3 different features of XGBoost.
Identify the model that performs best.

In [6]:
# explorimg different features of xgboost and tuning hyperparameters usingh stratified k-fold cross validation.
# explore 3 differnet features. 

XGB_Features = {
    "max_depth": [3, 5, 7],
    'scale_pos_weight': [1, 2, 3],
    'learning_rate': [0.01, 0.1, 0.2]}

xgb_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
xgb_random_search = RandomizedSearchCV(estimator=xgb_cv, param_distributions=XGB_Features, n_iter=10, cv=xgb_skf, scoring='f1', random_state=42)
xgb_random_search.fit(X, y)
print(f'Best hyperparameters: {xgb_random_search.best_params_}')
print(f'Best F1 score: {xgb_random_search.best_score_}')

Best hyperparameters: {'scale_pos_weight': 2, 'max_depth': 7, 'learning_rate': 0.1}
Best F1 score: 0.7284934652290077


### Tuning with GridSearchCV

Use the code cell below to set up your parameter grid and run GridSearchCV with your preferred model from above. You should tune 4-5 hyperparameters utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [7]:
# your code here

param_GS = {
    "max_depth": [3, 5, 7],
    'scale_pos_weight': [1, 2, 3],
    'learning_rate': [0.01, 0.1, 0.2]}
    
xgb_gs = GridSearchCV(estimator=xgb_cv, param_grid=param_GS, cv=xgb_skf, scoring='f1')
xgb_gs.fit(X, y)
print(f'Best hyperparameters: {xgb_gs.best_params_}')
print(f'Best F1 score: {xgb_gs.best_score_}')

Best hyperparameters: {'learning_rate': 0.2, 'max_depth': 5, 'scale_pos_weight': 2}
Best F1 score: 0.7288078457965205


### Tuning with RandomizedSearchCV

Using the code cell below as a starting point, tune your preferred model from above. Tune the same 4-5 hyperparameters from above utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [10]:
# tuning xgboost classifier with RandomizedSearchCV (tune more parameters than shown here)
param_dist = {
    "max_depth": [3, 5, 7],
    'scale_pos_weight': [1, 2, 3],
    'learning_rate': [0.01, 0.1, 0.2]}

# replace this placeholder model with your preferred model from above

xgb_random = RandomizedSearchCV(estimator=xgb_gs.best_estimator_,
                                param_distributions=param_dist, n_iter=20, cv=skf, scoring='f1', random_state=42)
xgb_random.fit(X_train, y_train)
print(f'Best parameters from RandomizedSearchCV: {xgb_random.best_params_}')
print(f'Best F1 score from RandomizedSearchCV: {xgb_random.best_score_}')   

# build your preferred model from above using best parameters from your RandomizedSearchCV
# and evaluate on the test set

xgb_random_best = XGBClassifier(random_state=42, scale_pos_weight=xgb_random.best_params_['scale_pos_weight'], 
                                max_depth=xgb_random.best_params_['max_depth'], 
                                learning_rate=xgb_random.best_params_['learning_rate'], 
                                enable_categorical=True)
xgb_random_best.fit(X_train, y_train)
y_pred_random = xgb_random_best.predict(X_test)
print(f'Classification report for RandomizedSearchCV-tuned model:\n{classification_report(y_test, y_pred_random)}')


Best parameters from RandomizedSearchCV: {'scale_pos_weight': 2, 'max_depth': 5, 'learning_rate': 0.2}
Best F1 score from RandomizedSearchCV: 0.7276089235024191
Classification report for RandomizedSearchCV-tuned model:
              precision    recall  f1-score   support

           0       0.93      0.88      0.90      7431
           1       0.67      0.80      0.73      2338

    accuracy                           0.86      9769
   macro avg       0.80      0.84      0.82      9769
weighted avg       0.87      0.86      0.86      9769



### Tuning with Optuna

Using the code cell below as a starting point, tune your preferred model from above. You should tune the same 4-5 parameters as above using cross validation. Train a final model using the best hyperparameters and report your model performance.

In [11]:
# tuning with Optuna (tune more parameters than shown here)
def objective(trial):
    scale_pos_weight = trial.suggest_float('scale_pos_weight', 1.0, 10.0)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
    
    # replace this placeholder model with your preferred model from above
    
    xgb_optuna = xgb_random_best
    
    cv_scores = cross_val_score(xgb_optuna, X, y, cv=skf, scoring='f1')
    return cv_scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f'Best parameters from Optuna: {study.best_params}')
print(f'Best F1 score from Optuna: {study.best_value}')

# build preferred model from above with best parameters from Optuna and evaluate on the test set
xgb_optuna_best = XGBClassifier(random_state=42, scale_pos_weight=study.best_params['scale_pos_weight'], 
                                  max_depth=study.best_params['max_depth'], 
                                  learning_rate=study.best_params['learning_rate'], 
                                  enable_categorical=True)
xgb_optuna_best.fit(X_train, y_train)
y_pred_optuna = xgb_optuna_best.predict(X_test)
print(f'Classification report for Optuna-tuned model:\n{classification_report(y_test, y_pred_optuna)}')


[I 2026-04-15 17:40:46,930] A new study created in memory with name: no-name-b090a2a9-237c-44e3-844e-f05c25082124
Best trial: 0. Best value: 0.728808:   5%|▌         | 1/20 [00:01<00:20,  1.09s/it]

[I 2026-04-15 17:40:48,028] Trial 0 finished with value: 0.7288078457965205 and parameters: {'scale_pos_weight': 4.223638576421689, 'max_depth': 9, 'learning_rate': 0.11875246500587175}. Best is trial 0 with value: 0.7288078457965205.


Best trial: 0. Best value: 0.728808:  10%|█         | 2/20 [00:02<00:20,  1.15s/it]

[I 2026-04-15 17:40:49,215] Trial 1 finished with value: 0.7288078457965205 and parameters: {'scale_pos_weight': 4.874201287235889, 'max_depth': 9, 'learning_rate': 0.18506486748788076}. Best is trial 0 with value: 0.7288078457965205.


Best trial: 0. Best value: 0.728808:  15%|█▌        | 3/20 [00:03<00:23,  1.37s/it]

...

[I 2026-04-15 17:41:14,671] Trial 17 finished with value: 0.7288078457965205 and parameters: {'scale_pos_weight': 1.4366913052111538, 'max_depth': 10, 'learning_rate': 0.09267340546939071}. Best is trial 0 with value: 0.7288078457965205.


Best trial: 0. Best value: 0.728808:  95%|█████████▌| 19/20 [00:29<00:01,  1.53s/it]

[I 2026-04-15 17:41:16,124] Trial 18 finished with value: 0.7288078457965205 and parameters: {'scale_pos_weight': 2.4746852844214455, 'max_depth': 7, 'learning_rate': 0.16174561426974926}. Best is trial 0 with value: 0.7288078457965205.


Best trial: 0. Best value: 0.728808: 100%|██████████| 20/20 [00:30<00:00,  1.54s/it]


[I 2026-04-15 17:41:17,661] Trial 19 finished with value: 0.7288078457965205 and parameters: {'scale_pos_weight': 6.896681079934546, 'max_depth': 3, 'learning_rate': 0.1322723938858804}. Best is trial 0 with value: 0.7288078457965205.
Best parameters from Optuna: {'scale_pos_weight': 4.223638576421689, 'max_depth': 9, 'learning_rate': 0.11875246500587175}
Best F1 score from Optuna: 0.7288078457965205
Classification report for Optuna-tuned model:
              precision    recall  f1-score   support

           0       0.95      0.81      0.88      7431
           1       0.60      0.87      0.71      2338

    accuracy                           0.83      9769
   macro avg       0.77      0.84      0.79      9769
weighted avg       0.87      0.83      0.84      9769



### Tuning results

In the markdown cell below describe your experience tuning with the different methods. Which produced the best results? Which do you prefer?


my grid search and optuna methods produced the same f1 score indicating that they worked equally as well, however each could have more paramters and features to improve that score. computationally it seems that gridsearch performs better however i have not throughoutly tested that and could be something to think about when choosing which method to tune models with. 